# 1. Imports and Setup
Project modules and required functions are imported. The system path is updated so `src` modules can be accessed.

In [2]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils import *
from src.hmm import HMM
from src.analysis import *

import math
import random
from scipy.stats import shapiro, ttest_ind, mannwhitneyu




# 2. Train HMM
A Hidden Markov Model (HMM) is initialized. Start and transition probabilities are set and emission probabilities are estimated from human CDS data.

Note: If you would like to use different data to train the model you can replace the file path assigned to the `filename` variable to the path to your file.

In [3]:
filename = "../data/GCF_000001405.40_GRCh38.p14_cds_from_genomic.fna.gz"
hmm = HMM()
hmm.initialize_parameters()
hmm.train_emission_probs_from_fasta(fasta_filename=filename, max_codons=None)

# 3. Small test sample
Analysis of a small test dataset I provided in `data/`. It contains 4 synthetic gene sequences such that the results expected are high adaptation scores for `Gene_1` and `Gene_2` and relatively lower adaptation scores for `Gene_3` and `Gene_4`.

In [4]:
test_file = "../data/test.fa"
gene_dict = load_fasta_dict(test_file)
results = analyze_genes(gene_dict, hmm, K=2)

for gene_id, data in results.items():
            print(f"Gene: {gene_id}")
            print(f"Average adaptation score: {data['avg_adaptation_score']}")

            for i, p in enumerate(data["paths"], start=1):
                print(f"Path {i}: {p['path']}")
                print(f"Adaptation score: {p['adaptation_score']}")
                print(f"Viterbi log-score: {p['viterbi_log_score']}")

            print()

Gene: Gene_1
Average adaptation score: 0.8
Path 1: AAAAAAAAAAAAAAAAANNN
Adaptation score: 0.85
Viterbi log-score: -86.05606091366525
Path 2: AAAAAAAAAAAAAAANNNNN
Adaptation score: 0.75
Viterbi log-score: -86.9032437297956

Gene: Gene_2
Average adaptation score: 0.8
Path 1: NNNNNAAAAAAAAAAAAAAA
Adaptation score: 0.75
Viterbi log-score: -85.74521088687197
Path 2: AANNNAAAAAAAAAAAAAAA
Adaptation score: 0.85
Viterbi log-score: -86.30250206004878

Gene: Gene_3
Average adaptation score: 0.125
Path 1: AAAAANNNNNNNNNNNNNNN
Adaptation score: 0.25
Viterbi log-score: -90.21405164252543
Path 2: NNNNNNNNNNNNNNNNNNNN
Adaptation score: 0.0
Viterbi log-score: -90.64763278258926

Gene: Gene_4
Average adaptation score: 0.3
Path 1: NNNNNNNNNNNNAAAAAAAA
Adaptation score: 0.4
Viterbi log-score: -89.69324006815799
Path 2: NNNNNNNNNNNNNNNNAAAA
Adaptation score: 0.2
Viterbi log-score: -89.778063506694



# 4. Synthetic Data

Two synthetic datasets are generated:
+ One biased toward human-like codon usage.
+ One generated from a uniform codon distribution.

Note: Sequences are created using codon sampling, with no fixed random seed, so results vary between runs.

In [11]:
def generate_sample_sequence(codons, probs, n):
    return "".join(random.choices(codons, weights=probs, k=n))
    
codon_list = generate_all_codons()
probs = [math.exp(hmm.emission_probs["A"][codon]) for codon in codon_list]

human_like_seq = {}
random_seq = {}

for i in range(1, 6):
    human_like_seq[f"h{i}"] = generate_sample_sequence(codon_list, probs, 20)
    random_seq[f"r{i}"] = generate_sample_sequence(codon_list, [1/64]*64, 20)


# 5. Test with synthetic data
The HMM is applied to the synthetic sequences, producing analysis results for each sequence. Average scores are calculated for human-like and random sequences.

In [12]:
results_human = analyze_genes(human_like_seq, hmm, K=2)
results_random = analyze_genes(random_seq, hmm, K=2)

human_scores = []
random_scores = []

for gene_id, data in results_human.items():
        human_scores.append(data["avg_adaptation_score"])

for gene_id, data in results_random.items():
        random_scores.append(data["avg_adaptation_score"])

avg_human_score = sum(human_scores)/len(results_human.keys())
avg_random_score = sum(random_scores)/len(results_random.keys())

print(f"Human average: {avg_human_score}")
print(f"Random average: {avg_random_score}")
print()


print("HUMAN SEQUENCES")

for gene_id, data in results_human.items():
            print(f"Gene: {gene_id}")
            print(f"Average adaptation score: {data['avg_adaptation_score']}")

            for i, p in enumerate(data["paths"], start=1):
                print(f"Path {i}: {p['path']}")
                print(f"Adaptation score: {p['adaptation_score']}")
                print(f"Viterbi log-score: {p['viterbi_log_score']}")

            print()

print("RANDOM SEQUENCES")

for gene_id, data in results_random.items():
            print(f"Gene: {gene_id}")
            print(f"Average adaptation score: {data['avg_adaptation_score']}")

            for i, p in enumerate(data["paths"], start=1):
                print(f"Path {i}: {p['path']}")
                print(f"Adaptation score: {p['adaptation_score']}")
                print(f"Viterbi log-score: {p['viterbi_log_score']}")

            print()

Human average: 0.7550000000000001
Random average: 0.33999999999999997

HUMAN SEQUENCES
Gene: h1
Average adaptation score: 0.825
Path 1: AAAAAAAAAAAAAAAAAAAA
Adaptation score: 1.0
Viterbi log-score: -89.57789618013693
Path 2: AAAAAAANNNNNNNAAAAAA
Adaptation score: 0.65
Viterbi log-score: -89.68261611574259

Gene: h2
Average adaptation score: 0.925
Path 1: AAAAAAAAAAAAAAAAAAAA
Adaptation score: 1.0
Viterbi log-score: -86.45349189098265
Path 2: NNNAAAAAAAAAAAAAAAAA
Adaptation score: 0.85
Viterbi log-score: -87.04555478087278

Gene: h3
Average adaptation score: 0.95
Path 1: AAAAAAAAAAAAAAAAAAAA
Adaptation score: 1.0
Viterbi log-score: -88.00017383882287
Path 2: AAAAANNAAAAAAAAAAAAA
Adaptation score: 0.9
Viterbi log-score: -88.08911105579294

Gene: h4
Average adaptation score: 0.2
Path 1: NNNNNNNNNNNNAAAAAAAA
Adaptation score: 0.4
Viterbi log-score: -90.57848640943801
Path 2: NNNNNNNNNNNNNNNNNNNN
Adaptation score: 0.0
Viterbi log-score: -90.64763278258926

Gene: h5
Average adaptation score:

# 6. Statistical Test
Normality of both score distributions is tested using the Shapiro-Wilk test. If both distributions are normal, a t-test is done, if atleast one is not normal, a Mann-Whitney U test is performed to compare the two groups and assess whether their difference is statistically significant.


In [13]:
_, p_human = shapiro(human_scores)
_, p_random = shapiro(random_scores)

print(f"Human p-value: {p_human}")
print(f"Random p-value: {p_random}")

if p_human and p_random > 0.05:
    _, p = ttest_ind(human_scores, random_scores)
    print("Both distributions are normal, so t-test was performed.")
    print(f"p-value: {p}")

else:
    _, p = mannwhitneyu(human_scores, random_scores, alternative="two-sided")
    print("Atleast one distribution is not normal, so Mann-Whitney U test was performed.")
    print(f"p-value: {p}")

Human p-value: 0.008035101579101405
Random p-value: 0.028291263116659388
Atleast one distribution is not normal, so Mann-Whitney U test was performed.
p-value: 0.09523809523809523
